# Juliet stratified cohort + Router suite 학습

Frozen split을 변경하지 않고 Train 6,000 / Dev 1,500을 Expert→CWE→leakage-group 다양성 우선으로 선택합니다. E6는 전량 보존합니다. 각 case의 후보와 5개 Expert 작업은 하나의 물리 API 요청으로 처리되며 case별로 체크포인트됩니다.

In [1]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
COHORT_CONFIG_PATH = EVAL_ROOT / 'configs' / 'cohort_15837.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
MAX_CONCURRENCY = 1000  # OpenRouter 한도에 맞춰 낮출 수 있음
TARGET_TRUTH_RECALL = 0.95
RUN_LEARNING_CURVES = False
MLP_BATCH_SIZE = 512
MLP_MAX_EPOCHS = 100
MLP_PATIENCE = 12
MLP_LEARNING_RATE = 2e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_DEVICE = 'auto'  # CUDA 가능 시 GPU, 아니면 CPU. 'cuda'로 강제 가능


In [2]:
import json, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.select_cohort import load_cohort_config, ensure_frozen_index, build_cohort_manifests
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.candidate_ranking import train_candidate_ranker_suite, rank_candidate_cache
from model_evaluation.workflow import (resolve_models, plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, train_utility_router, train_router_learning_curves)
from model_evaluation.adapters.llm_security import expert_assignments
from model_evaluation.diagnostics import calibrate_expert_confidence_thresholds

config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
cohort_config = load_cohort_config(COHORT_CONFIG_PATH)
models = resolve_models(ENV_FILE)
COHORT_DIR = EVAL_ROOT / 'work' / 'cohort_15837'
RUN_DIR = EVAL_ROOT / 'work' / 'router_training_stratified_7500'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_training_stratified_7500'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
CANDIDATE_RANKER_DIR = EVAL_ROOT / 'artifacts' / 'candidate_ranker'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
try:
    import torch
    print('MLP training device:', 'cuda' if torch.cuda.is_available() else 'cpu (CUDA unavailable)')
except ImportError:
    print('PyTorch is not installed')
print('Physical model:', models[0])

MLP training device: cuda
Physical model: deepseek/deepseek-v4-flash-0731


## 1. Leakage-safe stratified cohort manifest

In [3]:
index_report = ensure_frozen_index(config, mapping, progress=print)
cohort_report = build_cohort_manifests(config, cohort_config, output_directory=COHORT_DIR)
print(json.dumps(index_report, ensure_ascii=False, indent=2))
print(json.dumps(cohort_report, ensure_ascii=False, indent=2))

{
  "status": "reused",
  "index": "D:\\llm-security\\Model_Evaluation\\work\\index\\juliet.sqlite",
  "split_manifest": "D:\\llm-security\\Model_Evaluation\\work\\frozen\\split_manifest.json"
}
{
  "schema_version": "juliet-cohort-v1",
  "seed": 2026,
  "cohort_config_hash": "6f677c8c13ac1759333ce1b35d54ba7818a575da9e88029549fdb4dc04c88e4a",
  "frozen_split_manifest": "D:\\llm-security\\Model_Evaluation\\work\\frozen\\split_manifest.json",
  "frozen_split_manifest_sha256": "5e88026f39eeb365d92f6b58cabcf89d1e1ac76119b1fcc3bbbcef66d90d869a",
  "selection_inputs": "Expert/CWE/leakage-group metadata only; no candidate data",
  "splits": {
    "train": {
      "mode": "quota",
      "available_cases": 38563,
      "selected_cases": 6000,
      "manifest": "D:\\llm-security\\Model_Evaluation\\work\\cohort_15837\\cohort_train.jsonl",
      "manifest_sha256": "a11baaea12d9785712d3a56efdec1c0daa4ae02c4c364e87273052086de40f95",
      "expert_available": {
        "concurrency_toctou": 72,
     

## 2. Train/Dev materialization과 Semantic Analyzer cache

In [4]:
manifests = {split: COHORT_DIR / f'cohort_{split}.jsonl' for split in ('train', 'dev')}
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('train', 'dev'),
    selection_manifests=manifests, progress=print,
)
candidate_reports = {}
for split in ('train', 'dev'):
    candidate_reports[split] = cache_candidates(
        RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
    )
candidate_ranker_report = train_candidate_ranker_suite(
    train_cases=RUN_DIR / 'cases' / 'cases_train.jsonl',
    train_candidate_cache=RUN_DIR / 'candidates' / 'candidates_train.jsonl',
    dev_cases=RUN_DIR / 'cases' / 'cases_dev.jsonl',
    dev_candidate_cache=RUN_DIR / 'candidates' / 'candidates_dev.jsonl',
    artifact_dir=CANDIDATE_RANKER_DIR,
    report_path=RESULT_DIR / 'candidate_ranker_report.json', seed=config.seed,
)
ranked_candidate_paths = {}
for split in ('train', 'dev'):
    ranked_candidate_paths[split] = RUN_DIR / 'candidates' / f'candidates_{split}_ranked.jsonl'
    rank_candidate_cache(
        cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        input_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        output_path=ranked_candidate_paths[split],
        artifact_path=candidate_ranker_report['selected_artifact'],
    )
print(json.dumps({'materialization': materialization, 'candidates': candidate_reports, 'candidate_ranker': candidate_ranker_report}, ensure_ascii=False, indent=2))

{
  "materialization": {
    "mapping_hash": "57c2452af46fbd70f7e8e7416c1c5f76890f97276515e4a0cbd03acb4c56c719",
    "schema_version": "juliet-eval-v1",
    "seed": 2026,
    "splits": {
      "dev": {
        "attempted_cases": 1500,
        "available_indexed_cases": 8324,
        "cases": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\cases\\cases_dev.jsonl",
        "cwe_distribution": {
          "CWE-114": 61,
          "CWE-121": 47,
          "CWE-122": 45,
          "CWE-124": 34,
          "CWE-126": 34,
          "CWE-127": 39,
          "CWE-134": 68,
          "CWE-190": 59,
          "CWE-191": 54,
          "CWE-194": 49,
          "CWE-195": 50,
          "CWE-197": 47,
          "CWE-23": 48,
          "CWE-252": 88,
          "CWE-253": 89,
          "CWE-273": 18,
          "CWE-36": 67,
          "CWE-367": 18,
          "CWE-369": 48,
          "CWE-401": 33,
          "CWE-415": 31,
          "CWE-416": 23,
          "CWE-426": 48,
   

## 3. API 호출 계획

In [5]:
plans = {}
for split in ('train', 'dev'):
    plans[split] = plan_outcome_matrix(
        cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=ranked_candidate_paths[split],
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        model_ids=models, selection_policy='training_matrix',
        max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
print(json.dumps(plans, ensure_ascii=False, indent=2))

{
  "train": {
    "selection_contract_version": "candidate-selection-v2",
    "selection_policy": "training_matrix",
    "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\selections\\selected_train.jsonl",
    "selection_manifest_sha256": "24ca0dad23fb293cdb1beea61ccbeebce2af4b127b9c99ed72431262dfa949d6",
    "cases": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\cases\\cases_train.jsonl",
    "cases_sha256": "ad0a67a8b90fdfd1307ca9bc4e921f3f618646a5a20b936eab2e233da4662424",
    "candidate_cache": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\candidates\\candidates_train_ranked.jsonl",
    "candidate_cache_sha256": "cfeda15b23632136b1d8a8abd682ddb6e863c170c11723b6aead74f915aa721a",
    "max_candidates_per_case": 4,
    "case_count": 5982,
    "selected_candidate_count": 10482,
    "hard_negatives_per_case": 1,
    "positive_candidate_count": 4500,
    "hard_negative_candidate_c

## 4. Batched Expert outcome 수집 (case당 API 최대 1회, 최대 1,000건 비동기 동시 처리)

In [10]:
collection_reports = {}
for split in ('train', 'dev'):
    report = collect_outcome_matrix(
        env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=ranked_candidate_paths[split],
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / f'{split}_api_ledger.jsonl',
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
        model_ids=models, selection_policy='training_matrix',
        max_candidates_per_case=MAX_CANDIDATES_PER_CASE, max_concurrency=MAX_CONCURRENCY,
    )
    collection_reports[split] = report
    print(split, json.dumps(report, ensure_ascii=False, indent=2))
    if report['status'] != 'complete':
        print('실패한 case만 남았습니다. 성공한 case는 저장되었으며 이 셀을 다시 실행하면 실패 case부터 재개합니다.')
        break

train {
  "status": "complete",
  "stop_reason": null,
  "model": "deepseek/deepseek-v4-flash-0731",
  "selection_policy": "training_matrix",
  "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\selections\\selected_train.jsonl",
  "selection_manifest_sha256": "24ca0dad23fb293cdb1beea61ccbeebce2af4b127b9c99ed72431262dfa949d6",
  "cases_sha256": "ad0a67a8b90fdfd1307ca9bc4e921f3f618646a5a20b936eab2e233da4662424",
  "candidate_cache_sha256": "cfeda15b23632136b1d8a8abd682ddb6e863c170c11723b6aead74f915aa721a",
  "max_candidates_per_case": 4,
  "cases_seen": 5982,
  "completed_cases": 5982,
  "new_cases": 0,
  "failed_cases": 0,
  "max_concurrency": 1000,
  "scheduled_api_cases": 0,
  "case_level_retry_count": 0,
  "physical_requests_this_run": 0,
  "actual_cost_usd_this_run": 0.0,
  "request_contract": "one batched detection completion per case; transient failures may retry",
  "outcome_path": "D:\\llm-security\\Model_Evaluation\\work\\router_t

## 5. GPU Multi-task MLP + 기존 Escalation Gate

LR/GBDT는 실행하지 않습니다. 각 epoch의 learning rate와 train/validation loss가 바로 출력됩니다.

In [11]:
expected_ids = [item.assignment_id for item in expert_assignments(models)]
outcome_files = {split: RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl' for split in ('train', 'dev')}
audits = {
    split: audit_outcome_matrix(
        path, expected_assignment_ids=expected_ids,
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
    ) if path.exists() else {'complete': False, 'reason': 'missing'}
    for split, path in outcome_files.items()
}
print(json.dumps(audits, ensure_ascii=False, indent=2))
if all(item['complete'] for item in audits.values()):
    validator_calibration = calibrate_expert_confidence_thresholds(
        cases_path=RUN_DIR / 'cases' / 'cases_dev.jsonl',
        detection_path=outcome_files['dev'].with_suffix('.detections.jsonl'),
        report_path=RESULT_DIR / 'validator_calibration.json',
    )
    training_report = train_utility_router(
        train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
        train_selection_manifest=RUN_DIR / 'selections' / 'selected_train.jsonl',
        dev_selection_manifest=RUN_DIR / 'selections' / 'selected_dev.jsonl',
        train_cohort_manifest=manifests['train'],
        artifact_path=ARTIFACT, report_path=RESULT_DIR / 'training_report.json',
        model_ids=models, backends=('multitask_mlp',),
        seed=config.seed, target_truth_recall=TARGET_TRUTH_RECALL, mlp_device=MLP_DEVICE,
        mlp_batch_size=MLP_BATCH_SIZE, mlp_max_epochs=MLP_MAX_EPOCHS,
        mlp_patience=MLP_PATIENCE, mlp_learning_rate=MLP_LEARNING_RATE,
        mlp_weight_decay=MLP_WEIGHT_DECAY, progress=print,
    )
    training_report['validator_minimum_confidence_by_expert'] = validator_calibration['minimum_confidence_by_expert']
    training_report['candidate_ranker'] = {
        'report': str(RESULT_DIR / 'candidate_ranker_report.json'),
        'artifact': candidate_ranker_report['selected_artifact'],
        'artifact_sha256': candidate_ranker_report['selected_artifact_sha256'],
        'backend': candidate_ranker_report['selected_backend'],
        'selection_metric': candidate_ranker_report['selection_metric'],
        'feature_schema': candidate_ranker_report['feature_schema'],
        'dev_metrics': candidate_ranker_report['variants'][candidate_ranker_report['selected_backend']],
    }
    (RESULT_DIR / 'training_report.json').write_text(json.dumps(training_report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(training_report, ensure_ascii=False, indent=2))
    if RUN_LEARNING_CURVES:
        curve_report = train_router_learning_curves(
            train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
            train_cohort_manifest=manifests['train'],
            report_path=RESULT_DIR / 'learning_curves.json', seed=config.seed,
            target_truth_recall=TARGET_TRUTH_RECALL, backends=('multitask_mlp',),
            mlp_device=MLP_DEVICE, mlp_batch_size=MLP_BATCH_SIZE,
            mlp_max_epochs=MLP_MAX_EPOCHS, mlp_patience=MLP_PATIENCE,
            mlp_learning_rate=MLP_LEARNING_RATE, mlp_weight_decay=MLP_WEIGHT_DECAY,
            progress=print,
        )
        print(json.dumps(curve_report, ensure_ascii=False, indent=2))
else:
    print('Outcome matrix가 아직 완성되지 않았습니다. 4번 셀을 다시 실행해 남은 case를 수집하세요.')

{
  "train": {
    "row_count": 52410,
    "candidate_group_count": 10482,
    "expected_assignment_count": 5,
    "duplicate_row_count": 0,
    "incomplete_candidate_group_count": 0,
    "incomplete_preview": [],
    "expected_candidate_group_count": 10482,
    "missing_candidate_group_count": 0,
    "missing_preview": [],
    "unexpected_candidate_group_count": 0,
    "complete": true
  },
  "dev": {
    "row_count": 13595,
    "candidate_group_count": 2719,
    "expected_assignment_count": 5,
    "duplicate_row_count": 0,
    "incomplete_candidate_group_count": 0,
    "incomplete_preview": [],
    "expected_candidate_group_count": 2719,
    "missing_candidate_group_count": 0,
    "missing_preview": [],
    "unexpected_candidate_group_count": 0,
    "complete": true
  }
}
Router backend training: multitask_mlp
MLP training start | device=cuda | train=8738 | validation=1744 | batch_size=512 | max_epochs=100 | initial_lr=2.00e-03
epoch 001/100 | device=cuda | lr=2.00e-03 | train_loss=1